## download_weather_sources
Annual weather/hazard acquisition: downloads FEMA NRI (ArcGIS) and NOAA Climate Normals (NCEI) into their Unity Catalog Volumes (`{catalog}.raw.fema_nri`, `{catalog}.raw.climate_normals`). Thin composition root — all logic lives in the `data_fetch` package under `libs/`.

Runs `run_all(WEATHER_SOURCES)` — a SEPARATE tuple from the monthly `SOURCES`, so this never touches the monthly feeds. Batch policy is **abort-on-first**. Per-file audit is `{catalog}.audit.download_log`; this notebook writes one `pipeline_step_log` row via `StepLog`.

Design: `_dev_planning/design_docs/weather_sources_download_design.md`. Pull ALL available history — no windowing.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# Open the pipeline_step_log row for this download step. notebook_init injected Utils,
# StepLog, AUDIT, PIPELINE_RUN_ID. step_log_id is reused as the FK for every download_log row.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = 1,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = None,   # downloads land files in Volumes; per-file detail is in download_log
)

In [ ]:
# Composition root. notebook_init injected CATALOG, AUDIT, RAW_FILES, STATUS_*,
# PIPELINE_RUN_ID, spark, dbutils, datetime, timezone, StepLog. The data_fetch package is
# environment-agnostic; this cell wires the Databricks-specific collaborators.
import tempfile
from functools import partial

from data_fetch import (
    run_all, WEATHER_SOURCES, RunContext, VolumeFileWriter, DownloadJournal,
    DatabricksSecretResolver,
)
from pipeline_logging import download_log_insert, download_log_last_sha256

try:
    ctx = RunContext(
        catalog=CATALOG,
        pipeline_run_id=str(PIPELINE_RUN_ID),
        step_log_id=step.step_log_id,        # FK to the pipeline_step_log row opened above
        audit_schema=AUDIT,
        scratch_dir=tempfile.gettempdir(),   # serverless-safe; NEVER /local_disk0
        now=lambda: datetime.now(timezone.utc),
    )

    # abort-on-first: run_all raises on the first failed file (caught below -> step.fail).
    summary = run_all(
        WEATHER_SOURCES, ctx,
        writer=VolumeFileWriter(RAW_FILES),                      # base path from notebook_init
        journal=DownloadJournal(
            record=partial(download_log_insert, spark, AUDIT),
            last_sha256=partial(download_log_last_sha256, spark, AUDIT),
        ),
        secrets=DatabricksSecretResolver(dbutils, scope="marketpulse"),  # unused here (no keyed source)
    )
    print(summary.describe())

    step.rows_read    = len(summary.outcomes)                    # files attempted
    step.rows_written = len(summary.by_status(STATUS_SUCCEEDED)) # files newly landed
    step.succeed()

except Exception as e:
    step.fail(e); raise